# `ctc_loss`

These are my notes on CTCLoss. Some of this may be wrong because I was talking with LLMs to help me understand it.

According to the PyTorch documentation, [CTCLoss](https://docs.pytorch.org/docs/2.12/generated/torch.nn.CTCLoss.html) is a way of calculating the "loss between a continuous (unsegmented) time series and a target sequence."

But what it actually means by "continuous (unsegmented) time series" is really a discrete array, with an entry at each constant time step. For instance, let's say we have a recording of someone saying the word "hi". Perhaps the recording is one second long and we've broken it up into 0.1 second steps (probably would be better to use smaller steps, but I want this example to be simple), so there are 10 steps in the time series. Each step contains some array of numbers. This is our "continuous (unsegmented) time series", using somewhat non-intuitive definitions of the words "continuous" and "unsegmented".

Let's say we have a model which takes the time series as its input, and we want it to detect what letter is being spoken at each time step in the recording. In other words, we want to create a speech-to-text model that can recognize the word "hi". The output of the model will be a sequence of vectors, where each vector encodes the probability of each character. For simplicity, let's say we only have three possible characters indexed by different numbers: 0. a blank character, 1. "H", and 2. "I". The output of the model will have shape `[10, 3]`, since we have 10 steps in the input series and 3 possible characters. Our target sequence for the input of a person saying "hi" will be `tensor([1, 2])`, which resolves to `12 --> HI`.

Initially the model is untrained, so it may output random garbage at each of the 10 steps.

In [1]:
import torch

torch.manual_seed(2)

def argmax_stringify(t):
    map = {
        0: '_',
        1: 'H',
        2: 'I',
    }
    # One way to generate a sample from the model output probabilities is to
    # just take argmax, the most likely output. But later on we'll change to a
    # random sampler that matches the probabilities.
    return ''.join([map[idx] for idx in t.argmax(dim=-1).tolist()])

input_length = torch.tensor([10])
target_length = torch.tensor([2])

target_label = torch.tensor([1, 2]) # 12 --> HI


In [2]:
output_rand = torch.rand(10, 3)
output_rand /= output_rand.sum(dim=-1, keepdim=True)

print(output_rand)
print(argmax_stringify(output_rand))

torch.ctc_loss(
    output_rand.log(),
    target_label,
    input_length,
    target_length,
)

tensor([[0.3765, 0.2333, 0.3902],
        [0.2626, 0.3949, 0.3426],
        [0.3840, 0.0831, 0.5329],
        [0.0496, 0.4893, 0.4612],
        [0.2129, 0.4960, 0.2911],
        [0.4356, 0.4382, 0.1263],
        [0.2370, 0.7551, 0.0079],
        [0.2617, 0.2585, 0.4799],
        [0.2900, 0.4049, 0.3051],
        [0.3172, 0.3122, 0.3706]])
IHIHHHHIHI


tensor(5.6830)

Let's say that in one of the recordings in our training data, a voice is saying "H" for the first 3 time steps and "I" for the last 7. After the model has gone through some iterations of training, its output would hopefully end up being somewhere closer to a ideal, maybe something like:

In [3]:
output_ideal = torch.tensor([
    [0., 1, 0], # --> H
    [0, 1, 0], # --> H
    [0, 1, 0], # --> H
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
    [0, 0, 1], # --> I
])

print(argmax_stringify(output_ideal))

torch.ctc_loss(
    output_ideal.log(),
    target_label,
    input_length,
    target_length,
)

HHHIIIIIII


tensor(-0.)

There is not just one ideal model output for the recording, though. Any output which has a high probability of producing a sequence that represents the word "hi" is good. The CTC loss is a way of quantifying that probability.

The CTC loss is 0 for an ideal model output. But an actual output may never quite reach the ideal, and instead it might be something like the following when it's part-way through training:

In [4]:
output_training = torch.tensor([
    [0.56, 0.30, 0.14], # --> blank
    [0.16, 0.73, 0.11], # --> H
    [0.07, 0.87, 0.06], # --> H
    [0.40, 0.25, 0.35], # --> blank
    [0.49, 0.10, 0.41], # --> blank
    [0.11, 0.02, 0.87], # --> I
    [0.06, 0.04, 0.90], # --> I
    [0.11, 0.09, 0.80], # --> I
    [0.09, 0.02, 0.89], # --> I
    [0.48, 0.12, 0.40], # --> blank
])

print(argmax_stringify(output_training))

torch.ctc_loss(
    output_training.log(),
    target_label,
    input_length,
    target_length,
)

_HH__IIII_


tensor(1.2799)

The CTC loss for this output is somewhere between that of the ideal output and the random one.

While the model is being trained on some labelled data, it will back propagate the CTC loss score to adjust the parameters of the model.

Once the model is trained and ready to use, the output needs to be processed into actual words, using a simple collapsing rule: first, remove repeated consecutive tokens and, second, remove blanks. For instance: `_HH__IIIII_ --> _H_I_ --> HI`. CTC loss depends on this collapsing rule.

Argmax is not the only way to choose the tokens that we generate from the model's output probabilities. Instead, we can use `torch.multinomial` to generate a randomized token for each step, according to the probabilities that the model outputs.

In [5]:
def stringify_rand(t):
    indices = torch.multinomial(t, num_samples=1).flatten()
    map = {
        0: '_',
        1: 'H',
        2: 'I',
    }
    return ''.join([map[idx] for idx in indices.tolist()])

The ideal output that we chose above always produces the same result with this random sampling technique.

In [6]:
for _ in range(2):
    print(stringify_rand(output_ideal))

HHHIIIIIII
HHHIIIIIII


But a more realistic model output may produce different results each time.

In [7]:
for _ in range(2):
    print(stringify_rand(output_training))

H_H__IIII_
IHHIIIIII_


Given one particular model output probabilities array (like `output_training`), one particular sequence generated from it is called a path or trajectory. So `H_H__IIIIII` is one path and `HHHI_IIIII_` is another path.

Many different paths can resolve to the same label. For instance, `_HHH__II__` and `HH_IIIII__` would both resolve to `HI`.

In our example, the audio recording is labelled as representing the word "hi". So our target label is `HI`.

We can find the probability of generating one particular path $\mathbf \pi$ from the model output probabilities $\mathbf y$ by just multiplying the probabilities at each time step for the corresponding token in the path.

$$
P(\mathbf \pi) = \prod_{t=1}^T y^t_{\pi_t}
$$

$T$ is the length of the path (also the number of probability vectors in the model output).

$y^t_{\pi_t}$ is just the probability of generating token $\pi_t$ for timestep $t$, taken directly from the model output probabilities.

So for instance, if we want to find the probability of the path `HHHIIIIIII` (indices `1112222222`) from `output_training`:

In [8]:
path = torch.tensor([1, 1, 1, 2, 2, 2, 2, 2, 2, 2])

print(output_training[torch.arange(10), path].prod())

tensor(0.0061)


The probability of that particular path is fairly low.

In [9]:
print(output_ideal[torch.arange(10), path].prod())

tensor(1.)


`output_ideal` always produces that particular path, so its probability is exactly 1.

But as mentioned, there are many paths that satisfy the same target label. So we can calculate the probability of generating any path that resolves to the target label. If $\mathbf l$ is the target label and $x$ is the model input (in our case, the time series data representing the audio recording of someone saying "hi") which generates the output $y$, then the probably that feeding $x$ into the model to produce a path that sastisfies label $\mathbf l$ is:

$$
P(l | x) = \sum_{\pi \in B^{-1}(l)} P(\pi)
$$

$B(\pi)$ is the output of the collapsing rule when applied to a particular path $\pi$. So $\pi \in B^{-1}(l)$ is just the set of all paths which collapse to the target label $l$.

There are lots of ways to find all the paths that collapse to the target label, but we can use a brute force method for such a small example. Since there are 10 tokens with 3 possibilities, we have $3^{10} \approx 60,000$ possible paths.

In [10]:
import itertools

def collapse_path(path):
    return [a for a, _ in itertools.groupby(path) if a != 0]

def path_to_string(path):
    path_collapsed = collapse_path(path)
    map = {
        0: "_",
        1: "H",
        2: "I",
    }
    return ''.join(map[idx] for idx in path_collapsed)

def gen_matching_paths(num_outputs, num_tokens, target_label=None):
    for path in itertools.product(range(num_tokens), repeat=num_outputs):
        if target_label is not None:
            c = collapse_path(path)
            if c == target_label:
                # Just double check that all matching paths are 'HI'
                assert path_to_string(path) == path_to_string(target_label)
                yield path
        else:
            yield path

def sum_probs(model_output, paths):
    return model_output[torch.arange(model_output.shape[0]), paths].prod(dim=-1).sum()

all_paths = torch.tensor(list(gen_matching_paths(10, 3)))
hi_paths = torch.tensor(list(gen_matching_paths(10, 3, [1, 2])))

cases = [
    ('rand', output_rand),
    ('training', output_training),
    ('ideal', output_ideal),
]

for name, output in cases:
    print('\n')
    print(f"{name}: Prob of all paths = {sum_probs(output, all_paths):.4f}")
    print(f"{name}: Prob of 'HI' paths = {sum_probs(output, hi_paths):.4f}")




rand: Prob of all paths = 1.0000
rand: Prob of 'HI' paths = 0.0034


training: Prob of all paths = 1.0000
training: Prob of 'HI' paths = 0.2781


ideal: Prob of all paths = 1.0000
ideal: Prob of 'HI' paths = 1.0000


As we'd expect, for all three of our example model outpus, the probability of generating *any* path at all is 100% for all three example model outputs. The probability of generating a path that resolve to `HI` is the least for the random output, almost 0%, the most for the ideal output, exactly 100%. The partially trained output is somewhere in the middle, at about 30%.

In [11]:
hi_paths.shape[0] / all_paths.shape[0]

0.008382868465172992

Only about 0.8% of the possible paths resolve to "HI", and the random output is in that ballpark.

Now that we have a way to calculate the probability of generating the target label, the CTC loss is defined as:

$$
L = -\log P(l | x)
$$

In [12]:
def my_ctc_loss(log_probs, target_label):
    matching_paths = torch.tensor(list(gen_matching_paths(log_probs.shape[0], log_probs.shape[1], target_label)))
    prob = sum_probs(log_probs.exp(), matching_paths)
    return -prob.log()

for name, output in cases:
    print(f'\n{name}:')
    p = sum_probs(output, hi_paths)
    l = my_ctc_loss(output.log(), [1, 2])
    l_check = torch.ctc_loss(
        output.log(),
        target_label,
        input_length,
        target_length,
    )
    p_check = (-l_check).exp()

    print(f'  torch ctc_loss: {l_check:.4f}')
    print(f'  my ctc_loss:    {l:.4f}')
    print(f'  torch "HI" prob: {p:.4f}')
    print(f'  my "HI" prob:    {p_check:.4f}')



rand:
  torch ctc_loss: 5.6830
  my ctc_loss:    5.6830
  torch "HI" prob: 0.0034
  my "HI" prob:    0.0034

training:
  torch ctc_loss: 1.2799
  my ctc_loss:    1.2799
  torch "HI" prob: 0.2781
  my "HI" prob:    0.2781

ideal:
  torch ctc_loss: -0.0000
  my ctc_loss:    -0.0000
  torch "HI" prob: 1.0000
  my "HI" prob:    1.0000


Above, we're comparing the results for our example brute-force CTC loss calculation to those of PyTorch, for all three of our example model outputs, and they match up well.

But PyTorch doesn't generate all the possible paths and check each one to see if it generates the target label. There's a way to cut out much of that work, called the forward-backward algorithm, which is described in [this paper](https://www.cs.toronto.edu/~graves/icml_2006.pdf) in section 4.1.

I'm not going to describe the algorithm in detail because the paper does that well enough, and the following code is simple enough to understand. But it involves first modifying the target label to add a blank token at the beginning, end, and between each pair of tokens. The length of this modified sequence is $S$. Then we generate an array of shape `(T, S)`, where $T$ is the number of timesteps. Each element of this array depends on a corresponding element of the model output probabilities and up to three elements from the previous timestep.

In [13]:
def my_faster_ctc_loss(log_probs, target_label):
    # Interleave blanks into the target label
    l_prime = torch.zeros(2 * target_label.shape[0] + 1, dtype=torch.long)
    for i, token in enumerate(target_label):
        l_prime[2 * i + 1] = token

    output = log_probs.exp()

    T = output.shape[0]
    S = l_prime.shape[0]

    # Initialize the first row
    alpha = torch.zeros(T, S)
    alpha[0, 0] = output[0, 0]
    alpha[0, 1] = output[0, target_label[0]]

    for t in range(1, T):
        # Calculate each of the other rows sequentially
        for s in range(S):
            y = output[t, l_prime[s]]

            use_a_s1 = s >= 1
            use_a_s2 = (s >= 2) and l_prime[s] != 0 and l_prime[s-2] != l_prime[s]

            a_s  = alpha[t-1, s]
            a_s1 = alpha[t-1, s-1] if use_a_s1 else 0.0
            a_s2 = alpha[t-1, s-2] if use_a_s2 else 0.0

            alpha[t, s] = (a_s + a_s1 + a_s2) * y

    p = alpha[T-1, S-1] + alpha[T-1, S-2]

    ctc_loss = -(p).log()
    return ctc_loss

In [14]:
for name, output in cases:
    print(f'\n{name}:')
    p = sum_probs(output, hi_paths)
    l = my_faster_ctc_loss(output.log(), target_label)
    l_check = torch.ctc_loss(
        output.log(),
        target_label,
        input_length,
        target_length,
    )
    p_check = (-l_check).exp()

    print(f'  torch ctc_loss: {l_check:.4f}')
    print(f'  my ctc_loss:    {l:.4f}')


rand:
  torch ctc_loss: 5.6830
  my ctc_loss:    5.6830

training:
  torch ctc_loss: 1.2799
  my ctc_loss:    1.2799

ideal:
  torch ctc_loss: -0.0000
  my ctc_loss:    -0.0000


In the implementation above, we're doing a nested loop over all the timesteps and all the positions in the blank-interleaved target label. A GPU implementation of the forward-backward algorithm would still have to iterate over the timesteps, but it could parallelize over the target label size.